In [6]:
from google.colab import drive
import sys

drive.mount('/content/drive')

sys.path.append('/content/drive/MyDrive/Vasi')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import sys

# 1. Add the EXACT case-sensitive path
folder_path = '/content/drive/MyDrive/Vasi'
if folder_path not in sys.path:
    sys.path.append(folder_path)

# 2. Let's prove the files are actually visible to Colab
import os
print("Files Colab can see in this folder:")
print(os.listdir(folder_path))

Files Colab can see in this folder:
['inference_wrapper.py', 'saved_biobert_pico_model.zip', 'prompt_generator.py', '__pycache__', 'saved_biobert_pico_model']


In [5]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline, AutoModelForCausalLM


from inference_wrapper import predict_missing_pico
from prompt_generator import build_pico_prompt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


print("Loading BioBERT Encoder from Google Drive...")
model_path = "/content/drive/MyDrive/Vasi/saved_biobert_pico_model/"

encoder_tokenizer = AutoTokenizer.from_pretrained(model_path)
encoder_model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)

print("Loading BioMistral 7B Decoder...")
llm_model_name = "BioMistral/BioMistral-7B"

mistral_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
mistral_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    torch_dtype=torch.float16, # Keeps it fast and fits perfectly on Colab GPUs
    device_map="auto",
    use_safetensors=False
)

generator = pipeline(
    "text-generation",
    model=mistral_model,
    tokenizer=mistral_tokenizer
)

# ==========================================
# Phase 2: Run the End-to-End Pipeline
# ==========================================
print("\n" + "="*40)
print("RUNNING METHOD 1 PIPELINE")
print("="*40)

sample_abstract = (
    "We conducted a randomized controlled trial to evaluate the efficacy of a new "
    "cognitive behavioral therapy approach. The primary outcome measured was the "
    "reduction in severe anxiety symptoms after six months."
)

print(f"Input Abstract:\n{sample_abstract}\n")

# Step A: Encoder detects the missing element
prediction = predict_missing_pico(sample_abstract, encoder_model, encoder_tokenizer, device)
print(f"1. BioBERT Diagnosis: {prediction}")

# Step B: Handshake generates the strict prompt
prompt = build_pico_prompt(sample_abstract, prediction)
print("2. Prompt Generated Successfully.")

print("3. BioMistral Generating Response...\n")

# Step C: Decoder writes the question
llm_output = generator(
    prompt,
    max_new_tokens=75,
    do_sample=True,
    temperature=0.3,
    return_full_text=False
)

final_result = llm_output[0]['generated_text'].strip()

print("========================================")
print("FINAL OUTPUT TO AUTHORS:")
print("========================================")
print(final_result)

Loading BioBERT Encoder from Google Drive...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading BioMistral 7B Decoder...


pytorch_model.bin:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


RUNNING METHOD 1 PIPELINE
Input Abstract:
We conducted a randomized controlled trial to evaluate the efficacy of a new cognitive behavioral therapy approach. The primary outcome measured was the reduction in severe anxiety symptoms after six months.



Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=75) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. BioBERT Diagnosis: Missing P
2. Prompt Generated Successfully.
3. BioMistral Generating Response...

FINAL OUTPUT TO AUTHORS:
Could the authors clarify the patient population in this study?


In [6]:
!pip install biopython pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 104.4 MB/s eta 0:00:00


In [7]:
from Bio import Entrez
import pandas as pd


Entrez.email = "veranki2@illinois.edu"

print("1. Searching PubMed for fresh, unstructured clinical trials (2025-2026)...")

# Query: Clinical trials published recently, explicitly filtering OUT structured abstracts
search_query = '"clinical trial"[Publication Type] AND ("2025/01/01"[Date - Publication] : "2026/12/31"[Date - Publication]) NOT "structured abstract"[Filter]'

# Grabbing a small batch of 20 to test the PoC quickly
handle = Entrez.esearch(db="pubmed", term=search_query, retmax=20)
record = Entrez.read(handle)
handle.close()

id_list = record["IdList"]
print(f"Found {len(id_list)} recent IDs. Fetching the text...")

abstracts = []
if id_list:
    handle = Entrez.efetch(db="pubmed", id=id_list, retmode="xml")
    records = Entrez.read(handle)
    handle.close()

    for pubmed_article in records['PubmedArticle']:
        try:
            article = pubmed_article['MedlineCitation']['Article']
            if 'Abstract' in article:
                abstract_texts = article['Abstract']['AbstractText']
                # Join text to ensure it's a single unstructured block
                full_abstract = " ".join([str(text) for text in abstract_texts])

                # Basic sanity filter: must be a decent length, avoid obvious hardcoded headers
                if len(full_abstract) > 400 and "BACKGROUND:" not in full_abstract.upper():
                    abstracts.append(full_abstract)
        except Exception as e:
            continue

df_test = pd.DataFrame({"abstract": abstracts})
df_test.to_csv("/content/drive/MyDrive/Vasi/fresh_unstructured_test_data.csv", index=False)
print(f"Successfully saved {len(df_test)} fresh, unstructured abstracts to Google Drive!\n")




1. Searching PubMed for fresh, unstructured clinical trials (2025-2026)...
Found 20 recent IDs. Fetching the text...
Successfully saved 20 fresh, unstructured abstracts to Google Drive!



In [8]:
if len(df_test) > 0:
    test_abstract = df_test['abstract'].iloc[0]

    print("="*50)
    print("TESTING PIPELINE ON UNSEEN DATA")
    print("="*50)
    print(f"RAW ABSTRACT:\n{test_abstract}\n")

    # 1. BioBERT Encoder
    prediction = predict_missing_pico(test_abstract, encoder_model, encoder_tokenizer, device)
    print(f"--> BioBERT Diagnosis: Missing {prediction}")

    # 2. Handshake Prompt
    prompt = build_pico_prompt(test_abstract, prediction)

    # 3. BioMistral Decoder
    print("--> BioMistral Generating Question...\n")
    llm_output = generator(
        prompt,
        max_new_tokens=75,
        do_sample=True,
        temperature=0.3,
        return_full_text=False
    )

    print("========================================")
    print("FINAL OUTPUT TO AUTHORS:")
    print("========================================")
    print(llm_output[0]['generated_text'].strip())
else:
    print("No unstructured abstracts found in that small batch. Try increasing retmax.")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=75) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TESTING PIPELINE ON UNSEEN DATA
RAW ABSTRACT:
Key clinical features of neonatal acute respiratory distress syndrome (NARDS) are broadly comparable to those observed in pediatric and adult ARDS; however, evidence is insufficient to recommend high-frequency oscillatory ventilation (HFOV) or conventional mechanical ventilation (CMV) as the preferred first-line therapy. To evaluate whether HFOV is superior to CMV in reducing bronchopulmonary dysplasia (BPD) and other neonatal adverse outcomes, including death, among preterm infants (≤34 weeks' gestational age) with NARDS. This single-center randomized clinical trial conducted from August 1, 2019, to December 31, 2023, enrolled preterm infants born between 25 weeks 0 days and 34 weeks 6 days of gestation with NARDS who were stabilized with CMV. Data were analyzed from October to December 2024. Participants were randomly assigned to continue CMV or transition to elective HFOV. The primary outcome was BPD, assessed using 2 definitions: defini